In [ ]:
- ctrl+j 터미널에서 가상환경 만들기 => vscode에서 만들기
* python -m venv .venv
*가상환경 들어가기 : .venv\Scripts\activate
*pip 업그레이드 : python -m pip install --upgrade pip
*ipvnb파일 새로 만들기 : ex1_회귀모델저장 테스트 : ctrl+shift+p -> .venv선택 -> 커널 선택
*pip install statsmodels joblib flask

*pip freeze > requirements.txt
*require


SyntaxError: invalid syntax (3756961461.py, line 2)

In [1]:
print(1)

1


- 국토교통부 실거래가 공개 시스템 : https://rt.molit.go.kr/pt/xls/xls.do?&moblieAt=

In [25]:
import pandas as pd
import statsmodels.api as sm # 회귀모델
import joblib # 모델 저장

In [4]:
df = pd.read_csv('./data/trade_apt_api.csv', encoding='cp949')
df.sample()

,거래금액,건축년도,년,법정동,아파트,월,일,전용면적,지번,지역코드,층,해제사유발생일,해제여부
277,160000,1999,2021,명륜2가,아남3,1,22,172.17,237,11110,8,-,-


In [7]:
df.info

<bound method DataFrame.info of        거래금액  건축년도     년   법정동                    아파트  월   일      전용면적    지번  \
0     80000  2002  2021   신교동               신현(101동)  8  16   84.8200  6-13   
1    209000  2008  2021   사직동       광화문풍림스페이스본(106동)  8   5  163.3300   9-1   
2    160000  2008  2021   사직동  광화문풍림스페이스본(101동~105동)  8  10  158.9900     9   
3     96000  2008  2021   견지동                대성스카이렉스  8   4  116.0300   110   
4     32000  2003  2021   익선동                 현대뜨레비앙  8  14   48.5400    55   
..      ...   ...   ...   ...                    ... ..  ..       ...   ...   
313   86470  2000  2021   무악동                     현대  1  17   60.0000    82   
314   86500  2000  2021   무악동                     현대  1  17   60.0000    82   
315  135000  2019  2021   무악동               경희궁 롯데캐슬  1  22   59.6737    89   
316  120000  2008  2021   무악동                인왕산아이파크  1  27  114.9310    60   
317  109990  2000  2021   무악동                     현대  1  30   84.9200    82   

      지역코드   층 해제사유

In [8]:
# 회귀 모델
X = df[['건축년도', '전용면적', '층']].copy()
X['const'] = 1
y = df['거래금액']
X.shape, y.shape

((318, 4), (318,))

In [ ]:
model = sm.OLS(y, X).fit() # 회귀모델
model.summary()
# R-squared : X가 y를 설명해주는 수치
# Adj. R-squared : 조정된 r squared
# Durbin-Watson : X끼리의 상관이 있는지 수치(이상치는 2)
# coef(w와 b)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   거래금액   R-squared:                       0.648
Model:                            OLS   Adj. R-squared:                  0.644
Method:                 Least Squares   F-statistic:                     192.4
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           8.54e-71
Time:                        10:54:12   Log-Likelihood:                -3777.5
No. Observations:                 318   AIC:                             7563.
Df Residuals:                     314   BIC:                             7578.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
건축년도        1925.6916    212.616      9.057      0.000    1507.360    2344.023
전용면적         962.1507     47.367     20.313      0.000     868.955    1055.347
층           2058.1524    417.716      4.927      0.000    1236.276    2880.028
const      -3.855e+06   4.25e+05     -9.069      0.000   -4.69e+06   -3.02e+06
==============================================================================
Omnibus:                       20.985   Durbin-Watson:                   1.352
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               42.734
Skew:                           0.345   Prob(JB):                     5.25e-10
Kurtosis:                       4.658   Cond. No.                     4.33e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.33e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [10]:
X.iloc[0]

건축년도     2002.00
전용면적       84.82
층           1.00
const       1.00
Name: 0, dtype: float64

In [16]:
# 모델 예측하기
format(round(model.predict([[2002, 84, 12, 1]])[0]*10000), ',')

'1,056,398,580'

In [26]:
# 모델 저장
import os
if not os.path.exists('model'): # model
    os.mkdir('model')
joblib.dump(model, './model/ex1_apt_price_regression.joblib')

['./model/ex1_apt_price_regression.joblib']

In [27]:
def predict_apt_price(year, square, floor):
    r_model = joblib.load('./model/ex1_apt_price_regression.joblib')
    input_data = [[int(year), int(square), int(floor), 1]]
    result = round(r_model.predict(input_data)[0]*10000)
    return format(result, ',') + '원'
predict_apt_price(2002, 102, 8)

'1,147,259,615원'